In [1]:
import os

print(os.listdir("/content"))


['.config', 'hillstorm_no_indices.csv', 'sample_data']


In [2]:
import pandas as pd

df = pd.read_csv("/content/hillstorm_no_indices.csv")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

display(df.head())

Dataset shape: (64000, 12)

Column names:
['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'segment', 'visit', 'conversion', 'spend']


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


In [3]:
# Check the treatment groups and their sample counts

print("Treatment group counts:")
print(df["segment"].value_counts())

print("\nUnique treatment groups:")
print(df["segment"].unique())

Treatment group counts:
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

Unique treatment groups:
['Womens E-Mail' 'No E-Mail' 'Mens E-Mail']


In [4]:
# Select only Mens E-Mail and No E-Mail customers

df_dml = df[df["segment"].isin(["Mens E-Mail", "No E-Mail"])].copy()

# Create binary treatment variable
# 1 = Mens E-Mail
# 0 = No E-Mail

df_dml["treatment"] = (df_dml["segment"] == "Mens E-Mail").astype(int)

# Define outcome variable
# 1 = Customer converted
# 0 = Customer did not convert

df_dml["outcome"] = df_dml["conversion"].astype(int)

# Display sample information

print("DML Dataset Shape:", df_dml.shape)

print("\nTreatment distribution:")
print(df_dml["treatment"].value_counts())

print("\nOutcome distribution:")
print(df_dml["outcome"].value_counts())

print("\nSample preview:")
display(df_dml[["segment", "treatment", "conversion", "outcome"]].head(10))

DML Dataset Shape: (42613, 14)

Treatment distribution:
treatment
1    21307
0    21306
Name: count, dtype: int64

Outcome distribution:
outcome
0    42224
1      389
Name: count, dtype: int64

Sample preview:


,segment,treatment,conversion,outcome
1,No E-Mail,0,0,0
3,Mens E-Mail,1,0,0
8,Mens E-Mail,1,0,0
13,Mens E-Mail,1,0,0
14,No E-Mail,0,0,0
15,No E-Mail,0,0,0
16,Mens E-Mail,1,0,0
17,Mens E-Mail,1,0,0
18,Mens E-Mail,1,0,0
19,Mens E-Mail,1,0,0


In [5]:
# Select potential pre-treatment customer characteristics

features = [
    "recency",
    "history",
    "mens",
    "womens",
    "zip_code",
    "newbie",
    "channel"
]

X = df_dml[features].copy()

# Check data types
print("Feature data types:")
print(X.dtypes)

# Check missing values
print("\nMissing values:")
print(X.isnull().sum())

# Check unique values in categorical columns
print("\nZip code categories:")
print(X["zip_code"].value_counts())

print("\nChannel categories:")
print(X["channel"].value_counts())

# Display sample
print("\nFeature preview:")
display(X.head())

Feature data types:
recency       int64
history     float64
mens          int64
womens        int64
zip_code     object
newbie        int64
channel      object
dtype: object

Missing values:
recency     0
history     0
mens        0
womens      0
zip_code    0
newbie      0
channel     0
dtype: int64

Zip code categories:
zip_code
Surburban    19126
Urban        17105
Rural         6382
Name: count, dtype: int64

Channel categories:
channel
Web             18863
Phone           18567
Multichannel     5183
Name: count, dtype: int64

Feature preview:


,recency,history,mens,womens,zip_code,newbie,channel
1,6,329.08,1,1,Rural,1,Web
3,9,675.83,1,0,Rural,1,Web
8,9,675.07,1,1,Rural,1,Phone
13,2,101.64,0,1,Urban,0,Web
14,4,241.42,0,1,Rural,1,Multichannel


In [6]:
# Convert categorical variables into numerical columns

X_encoded = pd.get_dummies(
    X,
    columns=["zip_code", "channel"],
    drop_first=True,
    dtype=int
)

# Display the encoded features

print("Encoded dataset shape:", X_encoded.shape)

print("\nEncoded column names:")
print(X_encoded.columns.tolist())

print("\nData types:")
print(X_encoded.dtypes)

print("\nMissing values:")
print(X_encoded.isnull().sum())

print("\nEncoded feature preview:")
display(X_encoded.head())

Encoded dataset shape: (42613, 9)

Encoded column names:
['recency', 'history', 'mens', 'womens', 'newbie', 'zip_code_Surburban', 'zip_code_Urban', 'channel_Phone', 'channel_Web']

Data types:
recency                 int64
history               float64
mens                    int64
womens                  int64
newbie                  int64
zip_code_Surburban      int64
zip_code_Urban          int64
channel_Phone           int64
channel_Web             int64
dtype: object

Missing values:
recency               0
history               0
mens                  0
womens                0
newbie                0
zip_code_Surburban    0
zip_code_Urban        0
channel_Phone         0
channel_Web           0
dtype: int64

Encoded feature preview:


,recency,history,mens,womens,newbie,zip_code_Surburban,zip_code_Urban,channel_Phone,channel_Web
1,6,329.08,1,1,1,0,0,0,1
3,9,675.83,1,0,1,0,0,0,1
8,9,675.07,1,1,1,0,0,1,0
13,2,101.64,0,1,0,0,1,0,1
14,4,241.42,0,1,1,0,0,0,0


In [7]:
# Prepare DML input variables

X_final = X_encoded.copy()

T = df_dml["treatment"].copy()

Y = df_dml["outcome"].copy()

# Check dimensions
print("X shape:", X_final.shape)
print("T shape:", T.shape)
print("Y shape:", Y.shape)

# Check treatment distribution
print("\nTreatment distribution:")
print(T.value_counts())

# Check outcome distribution
print("\nOutcome distribution:")
print(Y.value_counts())

# Check whether EconML is installed
import importlib.util

if importlib.util.find_spec("econml") is not None:
    print("\nEconML is already installed.")
else:
    print("\nEconML is not installed yet.")

X shape: (42613, 9)
T shape: (42613,)
Y shape: (42613,)

Treatment distribution:
treatment
1    21307
0    21306
Name: count, dtype: int64

Outcome distribution:
outcome
0    42224
1      389
Name: count, dtype: int64

EconML is not installed yet.


In [8]:
!pip install econml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 69.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [econml]


In [9]:
import econml
print("EconML version:", econml.__version__)

from econml.dml import LinearDML
print("LinearDML imported successfully!")

EconML version: 0.17.0
LinearDML imported successfully!


In [10]:
# Import required libraries

from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# Create the DML estimator

dml_model = LinearDML(
    model_y=RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ),

    model_t=RandomForestClassifier(
        n_estimators=100,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ),

    discrete_treatment=True,
    cv=3,
    random_state=42
)

print("LinearDML estimator configured successfully!")
print(dml_model)

LinearDML estimator configured successfully!


In [11]:
# Train the Double Machine Learning model

print("Training DML model...")

dml_model.fit(
    Y,
    T,
    X=X_final
)

print("\nDML model training completed successfully!")

Training DML model...

DML model training completed successfully!


In [12]:
# Calculate Average Treatment Effect (ATE)

ate = dml_model.ate(X=X_final)

# Calculate 95% Confidence Interval
ate_lower, ate_upper = dml_model.ate_interval(
    X=X_final,
    alpha=0.05
)

print("========== DML RESULTS ==========")

print(f"Average Treatment Effect (ATE): {ate:.6f}")

print(f"95% Confidence Interval: [{ate_lower:.6f}, {ate_upper:.6f}]")

print(f"ATE in percentage points: {ate * 100:.4f}")

print("=================================")

========== DML RESULTS ==========
Average Treatment Effect (ATE): 0.006730
95% Confidence Interval: [0.004930, 0.008530]
ATE in percentage points: 0.6730


In [13]:
# Calculate Conditional Average Treatment Effects (CATE)

cate = dml_model.effect(X_final)

print("CATE shape:", cate.shape)

print("\nFirst 10 individual treatment effects:")
print(cate[:10])

print("\nCATE summary:")
print("Mean:", cate.mean())
print("Minimum:", cate.min())
print("Maximum:", cate.max())
print("Standard deviation:", cate.std())

CATE shape: (42613,)

First 10 individual treatment effects:
[0.01464709 0.00772967 0.01474541 0.00510598 0.00424069 0.00564306
 0.00451278 0.0049496  0.01516625 0.01022523]

CATE summary:
Mean: 0.0067300341686025115
Minimum: 0.001002189597781513
Maximum: 0.029542526705475535
Standard deviation: 0.003732812764183508


In [14]:
# Compare baseline characteristics between treatment and control groups

balance_check = df_dml.groupby("treatment")[
    ["recency", "history", "mens", "womens", "newbie"]
].mean().T

balance_check.columns = ["Control (No Email)", "Treatment (Mens Email)"]

balance_check["Difference"] = (
    balance_check["Treatment (Mens Email)"]
    - balance_check["Control (No Email)"]
)

print("Baseline characteristic comparison:")
display(balance_check)

# Treatment and outcome information

print("\nTreatment sample size:", len(T))
print("Outcome sample size:", len(Y))
print("Feature sample size:", len(X_final))

print("\nTreatment assignment rate:", T.mean())
print("Overall conversion rate:", Y.mean())

Baseline characteristic comparison:


,Control (No Email),Treatment (Mens Email),Difference
recency,5.749695,5.773642,0.023948
history,240.882653,242.835931,1.953278
mens,0.553224,0.550946,-0.002279
womens,0.547639,0.551415,0.003776
newbie,0.501971,0.501525,-0.000446



Treatment sample size: 42613
Outcome sample size: 42613
Feature sample size: 42613

Treatment assignment rate: 0.5000117335085538
Overall conversion rate: 0.009128669654800178


In [15]:
import pandas as pd

# Create Section A results summary

results = {
    "Metric": [
        "Treatment Variable",
        "Outcome Variable",
        "Treatment Group",
        "Control Group",
        "Sample Size",
        "Number of Features",
        "ATE",
        "ATE Percentage Points",
        "95% CI Lower",
        "95% CI Upper",
        "Mean CATE",
        "Minimum CATE",
        "Maximum CATE",
        "CATE Standard Deviation",
        "Treatment Assignment Rate",
        "Overall Conversion Rate"
    ],

    "Value": [
        "treatment",
        "outcome",
        "Mens E-Mail",
        "No E-Mail",
        len(T),
        X_final.shape[1],
        ate,
        ate * 100,
        ate_lower,
        ate_upper,
        cate.mean(),
        cate.min(),
        cate.max(),
        cate.std(),
        T.mean(),
        Y.mean()
    ]
}

results_df = pd.DataFrame(results)

display(results_df)

# Export results

results_df.to_csv("section_a_dml_results.csv", index=False)

print("\nSection A results saved successfully!")

,Metric,Value
0,Treatment Variable,treatment
1,Outcome Variable,outcome
2,Treatment Group,Mens E-Mail
3,Control Group,No E-Mail
4,Sample Size,42613
5,Number of Features,9
6,ATE,0.00673
7,ATE Percentage Points,0.673003
8,95% CI Lower,0.00493
9,95% CI Upper,0.00853



Section A results saved successfully!


In [16]:
from google.colab import files
files.download("section_a_dml_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
# Check the fitted DML model

print("Model fitted:", dml_model.fitted)

print("\nTreatment model:")
print(dml_model.models_t)

print("\nOutcome model:")
print(dml_model.models_y)

AttributeError: 'LinearDML' object has no attribute 'fitted'

In [18]:
# Check whether the DML model has been fitted

print("DML model fitted:", hasattr(dml_model, "models_y"))

print("\nTreatment models available:",
      hasattr(dml_model, "models_t"))

print("\nOutcome models available:",
      hasattr(dml_model, "models_y"))

print("\nNumber of CATE estimates:", len(cate))

print("\nATE:", ate)
print("95% Confidence Interval:", (ate_lower, ate_upper))

DML model fitted: True

Treatment models available: True

Outcome models available: True

Number of CATE estimates: 42613

ATE: 0.0067300341686025115
95% Confidence Interval: (np.float64(0.004930046744075232), np.float64(0.00853002159312979))


In [19]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    log_loss
)

# Obtain out-of-fold predictions from the fitted DML models

model_t = dml_model.models_t[0][0]
model_y = dml_model.models_y[0][0]

print("Treatment model:", type(model_t).__name__)
print("Outcome model:", type(model_y).__name__)

Treatment model: RandomForestClassifier
Outcome model: RandomForestRegressor


In [20]:
# Inspect available attributes for cross-fitting predictions

print("Treatment model attributes:")
print([a for a in dir(dml_model) if "pred" in a.lower() or "model" in a.lower()])

print("\nTreatment model structure:")
print(type(dml_model.models_t))
print("Number of folds:", len(dml_model.models_t))

print("\nOutcome model structure:")
print(type(dml_model.models_y))
print("Number of folds:", len(dml_model.models_y))

Treatment model attributes:
['_gen_model_final', '_gen_model_t', '_gen_model_y', '_gen_ortho_learner_model_final', '_gen_ortho_learner_model_nuisance', '_gen_rlearner_model_final', '_models_nuisance', '_ortho_learner_model_final', '_ortho_learner_model_nuisance', 'model_cate', 'model_final', 'model_final_', 'model_t', 'model_y', 'models_nuisance_', 'models_t', 'models_y', 'ortho_learner_model_final_', 'rlearner_model_final_']

Treatment model structure:
<class 'list'>
Number of folds: 1

Outcome model structure:
<class 'list'>
Number of folds: 1


In [21]:
# Inspect the models created during cross-fitting

print("Treatment model folds:", len(dml_model.models_t[0]))

print("Outcome model folds:", len(dml_model.models_y[0]))

print("\nTreatment models:")
for i, model in enumerate(dml_model.models_t[0]):
    print(f"Fold {i+1}:", type(model).__name__)

print("\nOutcome models:")
for i, model in enumerate(dml_model.models_y[0]):
    print(f"Fold {i+1}:", type(model).__name__)

Treatment model folds: 3
Outcome model folds: 3

Treatment models:
Fold 1: RandomForestClassifier
Fold 2: RandomForestClassifier
Fold 3: RandomForestClassifier

Outcome models:
Fold 1: RandomForestRegressor
Fold 2: RandomForestRegressor
Fold 3: RandomForestRegressor


In [22]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

# Recreate treatment model configuration
treatment_model = RandomForestClassifier(
    n_estimators=100,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

# Three-fold stratified cross-validation
skf = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# Generate out-of-fold treatment probabilities
treatment_predictions = cross_val_predict(
    treatment_model,
    X_final,
    T,
    cv=skf,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# Convert probabilities into treatment predictions
predicted_treatment = (treatment_predictions >= 0.5).astype(int)

# Calculate evaluation metrics
accuracy = accuracy_score(T, predicted_treatment)
auc = roc_auc_score(T, treatment_predictions)
loss = log_loss(T, treatment_predictions)

print("TREATMENT MODEL VALIDATION")
print("--------------------------")
print(f"Accuracy: {accuracy:.4f}")
print(f"ROC-AUC: {auc:.4f}")
print(f"Log Loss: {loss:.4f}")

TREATMENT MODEL VALIDATION
--------------------------
Accuracy: 0.5010
ROC-AUC: 0.5026
Log Loss: 0.6998


In [23]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
import numpy as np

# Configure outcome model
outcome_model = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

# Three-fold stratified cross-validation
skf = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# Generate out-of-fold predictions
outcome_predictions = cross_val_predict(
    outcome_model,
    X_final,
    Y,
    cv=skf,
    n_jobs=-1
)

# Calculate metrics
rmse = np.sqrt(mean_squared_error(Y, outcome_predictions))
mae = mean_absolute_error(Y, outcome_predictions)
r2 = r2_score(Y, outcome_predictions)

print("OUTCOME MODEL VALIDATION")
print("------------------------")
print(f"RMSE: {rmse:.6f}")
print(f"MAE: {mae:.6f}")
print(f"R2 Score: {r2:.6f}")

OUTCOME MODEL VALIDATION
------------------------
RMSE: 0.095623
MAE: 0.018208
R2 Score: -0.010879


In [24]:
# Consolidate model validation results

validation_results = pd.DataFrame({
    "Model": [
        "Treatment Model",
        "Treatment Model",
        "Treatment Model",
        "Outcome Model",
        "Outcome Model",
        "Outcome Model"
    ],
    "Metric": [
        "Accuracy",
        "ROC-AUC",
        "Log Loss",
        "RMSE",
        "MAE",
        "R2 Score"
    ],
    "Value": [
        accuracy,
        auc,
        loss,
        rmse,
        mae,
        r2
    ]
})

display(validation_results)

# Save validation results
validation_results.to_csv(
    "section_a_model_validation.csv",
    index=False
)

print("Validation results saved successfully!")

,Model,Metric,Value
0,Treatment Model,Accuracy,0.500974
1,Treatment Model,ROC-AUC,0.502577
2,Treatment Model,Log Loss,0.699822
3,Outcome Model,RMSE,0.095623
4,Outcome Model,MAE,0.018208
5,Outcome Model,R2 Score,-0.010879


Validation results saved successfully!


In [25]:
# Calculate conversion rates for each group

group_comparison = df_dml.groupby("treatment")["outcome"].agg(
    ["count", "sum", "mean"]
)

group_comparison.index = ["No E-Mail (Control)", "Mens E-Mail (Treatment)"]

group_comparison["conversion_percentage"] = (
    group_comparison["mean"] * 100
)

display(group_comparison)

# Calculate raw difference in conversion rates

control_rate = df_dml[df_dml["treatment"] == 0]["outcome"].mean()

treatment_rate = df_dml[df_dml["treatment"] == 1]["outcome"].mean()

raw_effect = treatment_rate - control_rate

print("\nRAW EXPERIMENTAL COMPARISON")
print("--------------------------")
print(f"Control conversion rate: {control_rate:.6f}")
print(f"Treatment conversion rate: {treatment_rate:.6f}")
print(f"Raw treatment effect: {raw_effect:.6f}")
print(f"Raw effect in percentage points: {raw_effect * 100:.4f}")
print(f"DML ATE: {ate:.6f}")
print(f"DML ATE in percentage points: {ate * 100:.4f}")

,count,sum,mean,conversion_percentage
No E-Mail (Control),21306,122,0.005726,0.572609
Mens E-Mail (Treatment),21307,267,0.012531,1.253109



RAW EXPERIMENTAL COMPARISON
--------------------------
Control conversion rate: 0.005726
Treatment conversion rate: 0.012531
Raw treatment effect: 0.006805
Raw effect in percentage points: 0.6805
DML ATE: 0.006730
DML ATE in percentage points: 0.6730


In [26]:
# Save raw treatment comparison

comparison_results = pd.DataFrame({
    "Metric": [
        "Control Sample Size",
        "Treatment Sample Size",
        "Control Conversions",
        "Treatment Conversions",
        "Control Conversion Rate",
        "Treatment Conversion Rate",
        "Raw Treatment Effect",
        "DML ATE",
        "Raw Effect Percentage Points",
        "DML Effect Percentage Points"
    ],
    "Value": [
        21306,
        21307,
        122,
        267,
        control_rate,
        treatment_rate,
        raw_effect,
        ate,
        raw_effect * 100,
        ate * 100
    ]
})

display(comparison_results)

comparison_results.to_csv(
    "section_a_raw_vs_dml_comparison.csv",
    index=False
)

print("Raw vs DML comparison saved successfully!")

,Metric,Value
0,Control Sample Size,21306.000000
1,Treatment Sample Size,21307.000000
2,Control Conversions,122.000000
3,Treatment Conversions,267.000000
4,Control Conversion Rate,0.005726
5,Treatment Conversion Rate,0.012531
6,Raw Treatment Effect,0.006805
7,DML ATE,0.006730
8,Raw Effect Percentage Points,0.680501
9,DML Effect Percentage Points,0.673003


Raw vs DML comparison saved successfully!


In [27]:
validation_results.to_csv(
    "section_a_model_validation.csv",
    index=False
)

In [28]:
comparison_results.to_csv(
    "section_a_raw_vs_dml_comparison.csv",
    index=False
)

In [29]:
from google.colab import files

files.download("section_a_model_validation.csv")
files.download("section_a_raw_vs_dml_comparison.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
# Generate all three Section A result files

results_df.to_csv("section_a_dml_results.csv", index=False)

validation_results.to_csv(
    "section_a_model_validation.csv", index=False
)

comparison_results.to_csv(
    "section_a_raw_vs_dml_comparison.csv", index=False
)

print("All three CSV files generated successfully!")

All three CSV files generated successfully!


In [32]:
from google.colab import files

files.download("section_a_dml_results.csv")
files.download("section_a_model_validation.csv")
files.download("section_a_raw_vs_dml_comparison.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>